In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/listing_sample.csv")

print(df.shape)
df.head()

(1000, 7)


,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1189451542,36612 Desert Tortoise Road,Murrieta,3.0,3.0,635000,"Welcome to 36612 Desert Tortoise Road, a beaut..."
1,1159634800,904 Amistad Court C,El Cajon,2.0,2.0,409000,PRICE REDUCED! on this IMMACULATE turnkey cond...
2,1144651476,1255 Delresto Drive,Beverly Hills,4.0,4.0,14495000,"Majestic Gated Italian Villa, An Architectural..."
3,1179077960,16855 Redwood Lodge Road,Los Gatos,4.0,3.0,1195000,Tucked into the trees of the Santa Cruz Mounta...
4,1190215696,5252 Agnes 2,Valley Village,2.0,2.0,599000,A magnificently remodeled corner unis with 3 (...


In [2]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumeric summary:")
display(df[["beds", "baths", "price"]].describe())

Shape: (1000, 7)

Columns:
['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price', 'remarks']

Missing values:
L_ListingID    0
L_Address      3
L_City         0
beds           1
baths          1
price          0
remarks        0
dtype: int64

Numeric summary:


,beds,baths,price
count,999.000000,999.000000,1.000000e+03
mean,3.361361,2.853854,1.519422e+06
std,1.412281,1.513024,2.721485e+06
min,0.000000,0.000000,1.500000e+04
25%,2.000000,2.000000,5.599750e+05
50%,3.000000,3.000000,8.170000e+05
75%,4.000000,3.000000,1.399000e+06
max,12.000000,16.000000,4.500000e+07


remarks: the listing description written by the agent. contains the natural-language text we’re going to analyze

In [3]:
df["remark_length"] = df["remarks"].str.len()

print(df["remark_length"].describe())

df[["remarks", "remark_length"]].head()

count    1000.000000
mean     1326.651000
std       574.222949
min        60.000000
25%       899.250000
50%      1281.000000
75%      1684.000000
max      3257.000000
Name: remark_length, dtype: float64


,remarks,remark_length
0,"Welcome to 36612 Desert Tortoise Road, a beaut...",1719
1,PRICE REDUCED! on this IMMACULATE turnkey cond...,506
2,"Majestic Gated Italian Villa, An Architectural...",2522
3,Tucked into the trees of the Santa Cruz Mounta...,808
4,A magnificently remodeled corner unis with 3 (...,1073


In [4]:
df["L_City"].value_counts().head(15)

L_City
Los Angeles         74
San Diego           37
Palm Springs        16
Temecula            14
Riverside           13
Irvine              13
San Jose            12
Palm Desert         12
Murrieta            11
Hemet               11
Rancho Cucamonga    11
Corona              11
Long Beach          10
Huntington Beach     9
Palmdale             8
Name: count, dtype: int64

## Common Word Analysis

To better understand the language used in real estate listing remarks, the text is converted to lowercase and tokenized into individual words. Common English stopwords such as "the", "and", and "is" are removed, along with punctuation and non-alphabetic tokens.

The remaining words are counted to identify the most frequently used real estate terms in the sample dataset.

In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

nltk.download("punkt")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

all_text = " ".join(df["remarks"].dropna().str.lower())

tokens = word_tokenize(all_text)

filtered_tokens = [
    word for word in tokens
    if word.isalpha() and word not in stop_words
]

word_freq = Counter(filtered_tokens)

word_freq.most_common(30)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/nghi_nguyen/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nghi_nguyen/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


[('home', 2451),
 ('living', 1869),
 ('room', 1371),
 ('space', 1116),
 ('offers', 1078),
 ('kitchen', 965),
 ('private', 884),
 ('dining', 795),
 ('features', 789),
 ('area', 721),
 ('property', 713),
 ('spacious', 679),
 ('bedrooms', 671),
 ('located', 619),
 ('access', 609),
 ('new', 573),
 ('garage', 569),
 ('community', 565),
 ('entertaining', 559),
 ('enjoy', 555),
 ('bedroom', 548),
 ('additional', 527),
 ('views', 522),
 ('one', 515),
 ('perfect', 482),
 ('primary', 481),
 ('large', 481),
 ('bathroom', 479),
 ('opportunity', 464),
 ('family', 460)]

## Bigram Analysis

Single words can miss important real estate concepts that are usually expressed as two-word phrases. Bigrams are pairs of consecutive words, such as "open floor", "floor plan", or "gated community".

The most frequent bigrams are extracted from the cleaned listing remarks to help identify candidate phrases for the Week 1 real estate taxonomy.

In [6]:
from nltk.util import ngrams

bigrams = list(ngrams(filtered_tokens, 2))

bigram_freq = Counter(bigrams)

bigram_freq.most_common(30)

[(('floor', 'plan'), 323),
 (('living', 'room'), 313),
 (('home', 'offers'), 303),
 (('primary', 'suite'), 290),
 (('living', 'space'), 278),
 (('natural', 'light'), 262),
 (('dining', 'area'), 189),
 (('home', 'office'), 177),
 (('square', 'feet'), 176),
 (('family', 'room'), 176),
 (('stainless', 'steel'), 172),
 (('everyday', 'living'), 171),
 (('shopping', 'dining'), 168),
 (('conveniently', 'located'), 162),
 (('steel', 'appliances'), 150),
 (('home', 'features'), 145),
 (('located', 'near'), 142),
 (('laundry', 'room'), 136),
 (('pool', 'spa'), 132),
 (('thoughtfully', 'designed'), 125),
 (('easy', 'access'), 125),
 (('bedrooms', 'bathrooms'), 122),
 (('living', 'area'), 120),
 (('living', 'spaces'), 119),
 (('living', 'entertaining'), 116),
 (('rare', 'opportunity'), 115),
 (('attached', 'garage'), 113),
 (('sq', 'ft'), 108),
 (('dining', 'room'), 105),
 (('living', 'dining'), 102)]

## Extended Bigram Review

To build a broader real estate taxonomy, the most frequent bigrams are expanded beyond the top 30. This provides a larger set of candidate phrases that can be reviewed and grouped into meaningful real estate categories.

In [7]:
top_bigrams = [
    (" ".join(pair), count)
    for pair, count in bigram_freq.most_common(150)
]

top_bigrams[:150]

[('floor plan', 323),
 ('living room', 313),
 ('home offers', 303),
 ('primary suite', 290),
 ('living space', 278),
 ('natural light', 262),
 ('dining area', 189),
 ('home office', 177),
 ('square feet', 176),
 ('family room', 176),
 ('stainless steel', 172),
 ('everyday living', 171),
 ('shopping dining', 168),
 ('conveniently located', 162),
 ('steel appliances', 150),
 ('home features', 145),
 ('located near', 142),
 ('laundry room', 136),
 ('pool spa', 132),
 ('thoughtfully designed', 125),
 ('easy access', 125),
 ('bedrooms bathrooms', 122),
 ('living area', 120),
 ('living spaces', 119),
 ('living entertaining', 116),
 ('rare opportunity', 115),
 ('attached garage', 113),
 ('sq ft', 108),
 ('dining room', 105),
 ('living dining', 102),
 ('step outside', 101),
 ('granite countertops', 100),
 ('quartz countertops', 97),
 ('property offers', 96),
 ('kitchen features', 95),
 ('recessed lighting', 95),
 ('covered patio', 92),
 ('full bathroom', 90),
 ('soaking tub', 89),
 ('generousl

## Taxonomy Candidate Exploration

The most frequent words and bigrams contain both useful real estate concepts and generic marketing language. To support taxonomy construction, candidate terms are grouped using real estate-related keywords.

These candidate groups will be manually reviewed before creating the final taxonomy.

In [8]:
# Combine frequent unigrams and bigrams into candidate terms
unigram_candidates = [
    (word, count)
    for word, count in word_freq.most_common(300)
]

bigram_candidates = [
    (" ".join(pair), count)
    for pair, count in bigram_freq.most_common(500)
]

all_candidates = unigram_candidates + bigram_candidates


category_keywords = {
    "rooms_spaces": [
        "room", "bedroom", "bathroom", "suite", "office",
        "kitchen", "dining", "laundry", "loft", "den"
    ],

    "kitchen_appliances": [
        "kitchen", "appliance", "stainless", "oven", "refrigerator",
        "dishwasher", "countertop", "cabinet", "island"
    ],

    "property_features": [
        "floor", "garage", "fireplace", "closet", "storage",
        "light", "window", "ceiling", "solar"
    ],

    "amenities": [
        "pool", "spa", "gym", "clubhouse", "community",
        "tennis", "security", "gated"
    ],

    "outdoor_features": [
        "yard", "patio", "balcony", "deck", "garden",
        "landscape", "lot", "outdoor"
    ],

    "location_access": [
        "school", "shopping", "dining", "freeway", "access",
        "near", "located", "park", "beach"
    ],

    "condition_style": [
        "new", "updated", "remodeled", "renovated", "modern",
        "luxury", "custom", "design"
    ],

    "measurements_financial": [
        "square", "sq", "feet", "acre", "price",
        "finance", "hoa", "solar"
    ]
}


taxonomy_candidates = {}

for category, keywords in category_keywords.items():
    matches = []

    for term, count in all_candidates:
        if any(keyword in term for keyword in keywords):
            matches.append((term, count))

    # Remove duplicates while keeping frequency order
    seen = set()
    unique_matches = []

    for term, count in sorted(matches, key=lambda x: x[1], reverse=True):
        if term not in seen:
            unique_matches.append((term, count))
            seen.add(term)

    taxonomy_candidates[category] = unique_matches[:40]


for category, terms in taxonomy_candidates.items():
    print(f"\n--- {category.upper()} ---")
    for term, count in terms:
        print(f"{term}: {count}")


--- ROOMS_SPACES ---
room: 1371
kitchen: 965
dining: 795
bedrooms: 671
bedroom: 548
bathroom: 479
residence: 437
suite: 409
bathrooms: 358
office: 322
living room: 313
primary suite: 290
laundry: 285
dining area: 189
home office: 177
family room: 176
shopping dining: 168
laundry room: 136
bedrooms bathrooms: 122
dining room: 105
living dining: 102
garden: 98
kitchen features: 95
full bathroom: 90
plenty room: 82
primary bedroom: 79
great room: 78
additional bedrooms: 76
residence offers: 70
chef kitchen: 69
formal dining: 68
updated kitchen: 62
bedroom bath: 59
bonus room: 58
bedroom bathroom: 57
suite offers: 57
remodeled kitchen: 56
bedroom full: 56
bedrooms full: 51
spacious bedrooms: 51

--- KITCHEN_APPLIANCES ---
kitchen: 965
appliances: 326
cabinetry: 288
countertops: 267
stainless: 188
stainless steel: 172
island: 169
steel appliances: 150
granite countertops: 100
cabinets: 98
quartz countertops: 97
kitchen features: 95
chef kitchen: 69
updated kitchen: 62
remodeled kitchen: 56